In [1]:
import os
import zipfile
import glob

# ZIP file name check pannu
zip_files = glob.glob("/content/*.zip")

print("ZIP files found:", zip_files)

if not zip_files:
    raise FileNotFoundError(
        "ZIP file kidaikkala. Colab-la ZIP file upload pannu."
    )

zip_path = zip_files[0]
extract_path = "/content/hinglish_dataset"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP extracted successfully!")

ZIP files found: ['/content/archive.zip']
ZIP extracted successfully!


In [2]:
for root, dirs, files in os.walk("/content/hinglish_dataset"):
    for file in files:
        print(os.path.join(root, file))

/content/hinglish_dataset/Eng - Hinglish/test.txt
/content/hinglish_dataset/Eng - Hinglish/train.txt
/content/hinglish_dataset/Eng - Hinglish/dev.txt


In [3]:
import pandas as pd
import glob
import os

all_txt_files = glob.glob(
    "/content/hinglish_dataset/**/*.txt",
    recursive=True
)

print("Text files:", all_txt_files)

Text files: ['/content/hinglish_dataset/Eng - Hinglish/test.txt', '/content/hinglish_dataset/Eng - Hinglish/train.txt', '/content/hinglish_dataset/Eng - Hinglish/dev.txt']


In [4]:

import os

base_path = "/content/hinglish_dataset/Eng - Hinglish"

for filename in ["train.txt", "dev.txt", "test.txt"]:
    filepath = os.path.join(base_path, filename)

    with open(filepath, "r", encoding="utf-8") as f:
        lines = f.readlines()

    print(f"\n{filename}: {len(lines)} lines")
    print("First 3 examples:")
    for line in lines[:3]:
        print(repr(line.strip()))



train.txt: 7040 lines
First 3 examples:
'Batman vs Superman\tbatman vs superman'
'The director is Zack Snyder , 27 % Rotten Tomatoes , 4.9 / 10 .\tZack Snyder director हैं , 27 % Rotten Tomatoes , 4.9 / 10 .'
'Not very popular it seems\tलगता है बहुत popular नहीं है'

dev.txt: 1020 lines
First 3 examples:
'Yeah , that was nice while it lasted . Seems to have dried up though\tहां , वह वाक्य में अछे हैं .. लगता हैं कि वह DRIED UP किये हैं'
'I live in Ohio and I hear about those " credits " for movies being made here . Normally I\'d watch Hugh Jackman in other movies .\tमें OHIO में रहने वाला हूँ , और मैं काफी US MOVIES का CREDITS सुना हूँ ... NORMALLY में ह्यू JACKMAN का MOVIES देखना पसन्द करता'
"He's a decent actor .\tवह DECENT ACTOR हैं"

test.txt: 942 lines
First 3 examples:
'hello\thello'
'hello there , I have not seen this movie so im going to take a minute to look it over :)\tहैलो यार , मैं इस movie को नहीं देख हों तो , तो मैं थोड़ी देर के लिए इसको देख लूंगा'
'Alright that is fine 

In [5]:

import os
import pandas as pd

base_path = "/content/hinglish_dataset/Eng - Hinglish"

def load_pairs(filename):
    pairs = []

    filepath = os.path.join(base_path, filename)

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")

            parts = line.split("\t", 1)

            if len(parts) == 2:
                source = parts[0].strip()
                target = parts[1].strip()

                if source and target:
                    pairs.append((source, target))

    return pd.DataFrame(pairs, columns=["english", "hinglish"])

train_df = load_pairs("train.txt")
dev_df = load_pairs("dev.txt")
test_df = load_pairs("test.txt")

print("Training pairs:", len(train_df))
print("Validation pairs:", len(dev_df))
print("Testing pairs:", len(test_df))

print("\nSample training data:")
display(train_df.head())

print("\nMissing values:")
print(train_df.isnull().sum())


Training pairs: 7040
Validation pairs: 1020
Testing pairs: 942

Sample training data:


,english,hinglish
0,Batman vs Superman,batman vs superman
1,"The director is Zack Snyder , 27 % Rotten Toma...","Zack Snyder director हैं , 27 % Rotten Tomatoe..."
2,Not very popular it seems,लगता है बहुत popular नहीं है
3,But the audiences liked it . It has a B cinema...,"but audience ने like किया , इसका cinema score ..."
4,Yes,yes



Missing values:
english     0
hinglish    0
dtype: int64


In [6]:

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Special tokens for decoder
START_TOKEN = "startseq"
END_TOKEN = "endseq"

# Add start/end tokens to target sentences
train_targets = [
    f"{START_TOKEN} {s} {END_TOKEN}" for s in train_df["hinglish"]
]
dev_targets = [
    f"{START_TOKEN} {s} {END_TOKEN}" for s in dev_df["hinglish"]
]
test_targets = [
    f"{START_TOKEN} {s} {END_TOKEN}" for s in test_df["hinglish"]
]

# Build source and target tokenizers using training data only
src_tokenizer = Tokenizer(filters="", oov_token="<unk>")
tgt_tokenizer = Tokenizer(filters="", oov_token="<unk>")

src_tokenizer.fit_on_texts(train_df["english"])
tgt_tokenizer.fit_on_texts(train_targets)

# Convert text to integer sequences
src_train = src_tokenizer.texts_to_sequences(train_df["english"])
src_dev = src_tokenizer.texts_to_sequences(dev_df["english"])
src_test = src_tokenizer.texts_to_sequences(test_df["english"])

tgt_train = tgt_tokenizer.texts_to_sequences(train_targets)
tgt_dev = tgt_tokenizer.texts_to_sequences(dev_targets)
tgt_test = tgt_tokenizer.texts_to_sequences(test_targets)

# Determine sequence lengths from training data
MAX_SRC_LEN = max(len(seq) for seq in src_train)
MAX_TGT_LEN = max(len(seq) for seq in tgt_train)

# Pad source and target sequences
X_train = pad_sequences(src_train, maxlen=MAX_SRC_LEN, padding="post")
X_dev = pad_sequences(src_dev, maxlen=MAX_SRC_LEN, padding="post")
X_test = pad_sequences(src_test, maxlen=MAX_SRC_LEN, padding="post")

Y_train_full = pad_sequences(
    tgt_train, maxlen=MAX_TGT_LEN, padding="post"
)
Y_dev_full = pad_sequences(
    tgt_dev, maxlen=MAX_TGT_LEN, padding="post"
)
Y_test_full = pad_sequences(
    tgt_test, maxlen=MAX_TGT_LEN, padding="post"
)

# Teacher forcing:
# decoder input = target without final token
# decoder output = target without initial token
decoder_train = Y_train_full[:, :-1]
decoder_target_train = Y_train_full[:, 1:]

decoder_dev = Y_dev_full[:, :-1]
decoder_target_dev = Y_dev_full[:, 1:]

SRC_VOCAB_SIZE = len(src_tokenizer.word_index) + 1
TGT_VOCAB_SIZE = len(tgt_tokenizer.word_index) + 1

print("Source vocabulary:", SRC_VOCAB_SIZE)
print("Target vocabulary:", TGT_VOCAB_SIZE)
print("Max English length:", MAX_SRC_LEN)
print("Max target length:", MAX_TGT_LEN)
print("Encoder input shape:", X_train.shape)
print("Decoder input shape:", decoder_train.shape)
print("Decoder target shape:", decoder_target_train.shape)


Source vocabulary: 5655
Target vocabulary: 7680
Max English length: 283
Max target length: 373
Encoder input shape: (7040, 283)
Decoder input shape: (7040, 372)
Decoder target shape: (7040, 372)


In [7]:

import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.models import Model

# Hyperparameters
EMBED_DIM = 128
LATENT_DIM = 256

# Encoder
encoder_inputs = Input(shape=(MAX_SRC_LEN,), name="encoder_input")

encoder_embedding = Embedding(
    input_dim=SRC_VOCAB_SIZE,
    output_dim=EMBED_DIM,
    mask_zero=True,
    name="encoder_embedding"
)(encoder_inputs)

encoder_lstm = LSTM(
    LATENT_DIM,
    return_state=True,
    name="encoder_lstm"
)

_, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(MAX_TGT_LEN - 1,), name="decoder_input")

decoder_embedding_layer = Embedding(
    input_dim=TGT_VOCAB_SIZE,
    output_dim=EMBED_DIM,
    mask_zero=True,
    name="decoder_embedding"
)

decoder_embedding = decoder_embedding_layer(decoder_inputs)

decoder_lstm = LSTM(
    LATENT_DIM,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

decoder_dense = Dense(
    TGT_VOCAB_SIZE,
    activation="softmax",
    name="word_predictions"
)

decoder_outputs = decoder_dense(decoder_outputs)

model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"]
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, 283)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None, 372)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 283, 128)  │    723,840 │ encoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 283)       │          0 │ encoder_input[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 372, 128)  │    983,040 │ decoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 256),     │    394,240 │ encoder_embeddin… │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 372,      │    394,240 │ decoder_embeddin… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_predictions    │ (None, 372, 7680) │  1,973,760 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,469,120 (17.05 MB)

 Trainable params: 4,469,120 (17.05 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
# Encoder inference model
encoder_model = tf.keras.Model(
    encoder_inputs,
    encoder_states
)

# Decoder inference inputs
decoder_state_input_h = tf.keras.Input(shape=(LATENT_DIM,))
decoder_state_input_c = tf.keras.Input(shape=(LATENT_DIM,))

decoder_states_inputs = [
    decoder_state_input_h,
    decoder_state_input_c
]

decoder_single_input = tf.keras.Input(shape=(1,))

decoder_single_embedding = decoder_embedding_layer(
    decoder_single_input
)

decoder_single_output, state_h, state_c = decoder_lstm(
    decoder_single_embedding,
    initial_state=decoder_states_inputs
)

decoder_single_output = decoder_dense(decoder_single_output)

decoder_model = tf.keras.Model(
    [decoder_single_input] + decoder_states_inputs,
    [decoder_single_output, state_h, state_c]
)

print("Inference models ready!")

Inference models ready!


In [13]:
import numpy as np

index_to_word = {
    index: word
    for word, index in tgt_tokenizer.word_index.items()
}

def translate_sentence(sentence):
    seq = src_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=MAX_SRC_LEN, padding="post")

    states = encoder_model.predict(seq, verbose=0)

    start_id = tgt_tokenizer.word_index["startseq"]
    end_id = tgt_tokenizer.word_index["endseq"]

    target_seq = np.array([[start_id]])
    output_words = []

    for _ in range(MAX_TGT_LEN):
        predictions, h, c = decoder_model.predict(
            [target_seq] + states, verbose=0
        )

        predicted_id = int(np.argmax(predictions[0, -1, :]))

        if predicted_id == end_id or predicted_id == 0:
            break

        word = index_to_word.get(predicted_id, "<unk>")
        output_words.append(word)

        target_seq = np.array([[predicted_id]])
        states = [h, c]

    return " ".join(output_words)

sentences = [
    "How are you?",
    "I like this movie.",
    "What is your name?",
    "Today is a good day.",
    "Thank you very much."
]

for sentence in sentences:
    print("English:", sentence)
    print("Hinglish:", translate_sentence(sentence))
    print()

English: How are you?
Hinglish: 

English: I like this movie.
Hinglish: मुझे , है

English: What is your name?
Hinglish: मुझे है

English: Today is a good day.
Hinglish: मुझे , है

English: Thank you very much.
Hinglish: मुझे है

